# 🎓 ERIA — Education Regulation Impact Analyzer
### Simplifying Education Policies and Circulars for Every Institution

**Sources:** UGC Notices, Regulations, Circulars, Guidelines  
**Tech Stack:** PyMuPDF · BeautifulSoup · Gemini API · Gradio  
**Timeline:** 14 days | **Domain:** EdTech / Education Policy Analytics

---
### Notebook Structure
1. Install Dependencies
2. Configuration & API Setup
3. Web Scraper — UGC Portal PDF Downloader
4. PDF Ingestion & Text Preprocessing
5. Document Classifier
6. AI Analysis Pipeline (Gemini)
7. Impact Analyzer (Short / Medium / Long Term)
8. Stakeholder Mapper
9. Chronology Builder
10. Gradio Dashboard
11. Export & Deploy to Hugging Face Spaces

## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q pymupdf pdfplumber
!pip install -q google-generativeai
!pip install -q gradio
!pip install -q beautifulsoup4 requests lxml
!pip install -q spacy nltk
!pip install -q fpdf2 pandas tqdm
!python -m spacy download en_core_web_sm -q

print('✅ All dependencies installed successfully!')

## ⚙️ Cell 2 — Configuration & API Setup

In [ ]:
import os
import json
import time
import requests
import re
from pathlib import Path
from google.colab import userdata

# ── Gemini API Key ──────────────────────────────────────────────────────────
# Add your key in Colab → Secrets (left panel 🔑) with name: GEMINI_API_KEY
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ Gemini API key loaded from Colab Secrets')
except Exception:
    GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'  # fallback — replace this
    print('⚠️  Using hardcoded API key — move to Colab Secrets for safety')

import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

# ── Directory Setup ─────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/ERIA')
PDF_DIR     = BASE_DIR / 'input_pdfs'
OUTPUT_DIR  = BASE_DIR / 'outputs'
REPORT_DIR  = BASE_DIR / 'reports'

for d in [PDF_DIR, OUTPUT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── UGC Source URLs ─────────────────────────────────────────────────────────
UGC_SOURCES = {
    'notices'     : 'https://www.ugc.gov.in/Notices',
    'regulations' : 'https://www.ugc.gov.in/regulations',
    'circulars'   : 'https://www.ugc.gov.in/Circulars',
    'guidelines'  : 'https://www.ugc.gov.in/Guideline',
}

print('✅ Configuration complete')
print(f'   PDF folder  : {PDF_DIR}')
print(f'   Output folder: {OUTPUT_DIR}')

## 🌐 Cell 3 — UGC Portal Web Scraper & PDF Downloader

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from tqdm import tqdm

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

def fetch_page(url, retries=3):
    """Fetch a web page with retry logic."""
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            return response
        except requests.RequestException as e:
            print(f'   Attempt {attempt+1} failed: {e}')
            time.sleep(2)
    return None

def extract_pdf_links(url, source_name):
    """Extract all PDF links from a UGC page."""
    print(f'\n🔍 Scanning: {source_name} → {url}')
    response = fetch_page(url)
    if not response:
        print(f'   ❌ Could not fetch {url}')
        return []

    soup = BeautifulSoup(response.text, 'lxml')
    pdf_links = []

    # Find all anchor tags with PDF references
    for tag in soup.find_all('a', href=True):
        href = tag['href'].strip()
        # Match direct .pdf links or UGC document paths
        if href.lower().endswith('.pdf') or '/pdfnews/' in href or '/Doc/' in href:
            full_url = urljoin(url, href)
            title = tag.get_text(strip=True) or Path(href).stem
            title = re.sub(r'[^\w\s-]', '', title)[:80].strip()
            pdf_links.append({'url': full_url, 'title': title, 'source': source_name})

    # Also look inside tables (UGC uses table layouts)
    for row in soup.find_all('tr'):
        cells = row.find_all('td')
        for cell in cells:
            link = cell.find('a', href=True)
            if link:
                href = link['href'].strip()
                if href.lower().endswith('.pdf') or '/pdfnews/' in href:
                    full_url = urljoin(url, href)
                    title = link.get_text(strip=True)[:80]
                    if {'url': full_url} not in [{'url': x['url']} for x in pdf_links]:
                        pdf_links.append({'url': full_url, 'title': title, 'source': source_name})

    print(f'   Found {len(pdf_links)} PDF links')
    return pdf_links

def download_pdf(pdf_info, save_dir, max_size_mb=10):
    """Download a single PDF file."""
    url   = pdf_info['url']
    title = re.sub(r'\s+', '_', pdf_info['title'])[:60]
    fname = f"{pdf_info['source']}_{title}.pdf"
    fpath = save_dir / fname

    if fpath.exists():
        return str(fpath)  # skip already downloaded

    try:
        r = requests.get(url, headers=HEADERS, timeout=20, stream=True)
        r.raise_for_status()
        size = 0
        with open(fpath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                size += len(chunk)
                if size > max_size_mb * 1024 * 1024:
                    print(f'   ⚠️  Skipping large file: {fname}')
                    fpath.unlink(missing_ok=True)
                    return None
        return str(fpath)
    except Exception as e:
        print(f'   ❌ Download failed for {fname}: {e}')
        return None

def scrape_and_download_ugc(max_per_source=10):
    """Scrape all UGC sources and download PDFs."""
    all_links = []
    for name, url in UGC_SOURCES.items():
        links = extract_pdf_links(url, name)
        all_links.extend(links[:max_per_source])  # limit per source

    print(f'\n📥 Downloading {len(all_links)} PDFs...')
    downloaded = []
    for info in tqdm(all_links):
        path = download_pdf(info, PDF_DIR)
        if path:
            downloaded.append({'path': path, **info})
        time.sleep(0.5)  # be polite to the server

    print(f'\n✅ Downloaded {len(downloaded)} PDFs to {PDF_DIR}')
    return downloaded

# ── Run Scraper ─────────────────────────────────────────────────────────────
# Set max_per_source to control how many PDFs to download per section
downloaded_files = scrape_and_download_ugc(max_per_source=5)

# ── OR: Manually upload your own PDFs to Colab ──────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# for fname, data in uploaded.items():
#     with open(PDF_DIR / fname, 'wb') as f: f.write(data)

## 📄 Cell 4 — PDF Ingestion & Text Preprocessing

In [ ]:
import fitz  # PyMuPDF
import pdfplumber
import pandas as pd

def extract_text_pymupdf(pdf_path):
    """Extract text using PyMuPDF — best for digital PDFs."""
    text = ''
    try:
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text('text') + '\n'
        doc.close()
    except Exception as e:
        print(f'PyMuPDF error: {e}')
    return text.strip()

def extract_text_pdfplumber(pdf_path):
    """Extract text using pdfplumber — better for tables and structured PDFs."""
    text = ''
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + '\n'
    except Exception as e:
        print(f'pdfplumber error: {e}')
    return text.strip()

def clean_text(text):
    """Clean extracted text: remove noise, fix spacing, normalize."""
    # Remove excessive whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    # Remove page numbers like '- 3 -' or 'Page 3 of 12'
    text = re.sub(r'-\s*\d+\s*-', '', text)
    text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)
    # Remove footer lines with phone/fax numbers
    text = re.sub(r'\d{3}[-.]\d{3}[-.]\d{4}', '', text)
    # Normalize unicode dashes
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    return text.strip()

def extract_metadata(text, filename):
    """Extract structured metadata from document text."""
    meta = {
        'filename'    : filename,
        'issuing_body': 'Unknown',
        'doc_number'  : '',
        'date'        : '',
        'subject'     : '',
        'word_count'  : len(text.split()),
    }

    # Detect issuing body
    bodies = {
        'University Grants Commission': 'UGC',
        'UGC': 'UGC',
        'AICTE': 'AICTE',
        'NAAC': 'NAAC',
        'Ministry of Education': 'MoE',
        'National Institute of Disaster Management': 'NIDM',
        'Election Commission': 'ECI',
    }
    for keyword, body in bodies.items():
        if keyword.lower() in text.lower():
            meta['issuing_body'] = body
            break

    # Extract document/file number (e.g. F.No. 3-1/2004)
    num_match = re.search(r'[FD]\.?\s*No\.?\s*[\w/-]+', text)
    if num_match:
        meta['doc_number'] = num_match.group(0).strip()

    # Extract date
    date_match = re.search(
        r'(\d{1,2}\s+(?:January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{4})',
        text, re.IGNORECASE
    )
    if date_match:
        meta['date'] = date_match.group(1)

    # Extract subject line
    sub_match = re.search(r'[Ss]ubject\s*[:\-]\s*(.{20,200})', text)
    if sub_match:
        meta['subject'] = sub_match.group(1).strip()[:200]

    return meta

def chunk_text(text, max_chars=12000):
    """Split large text into chunks for Gemini's context window."""
    paragraphs = text.split('\n\n')
    chunks, current = [], ''
    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += para + '\n\n'
        else:
            if current:
                chunks.append(current.strip())
            current = para + '\n\n'
    if current:
        chunks.append(current.strip())
    return chunks if chunks else [text[:max_chars]]

def process_pdf(pdf_path):
    """Full pipeline: load → extract → clean → chunk."""
    path = Path(pdf_path)
    print(f'  Processing: {path.name}')

    # Try PyMuPDF first, fallback to pdfplumber
    raw_text = extract_text_pymupdf(str(path))
    if len(raw_text) < 100:
        raw_text = extract_text_pdfplumber(str(path))

    if len(raw_text) < 50:
        return None

    clean = clean_text(raw_text)
    meta  = extract_metadata(clean, path.name)
    chunks = chunk_text(clean)

    return {
        'path'    : str(path),
        'raw_text': raw_text,
        'text'    : clean,
        'chunks'  : chunks,
        'metadata': meta,
    }

# ── Process all downloaded PDFs ─────────────────────────────────────────────
pdf_files = list(PDF_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDFs in {PDF_DIR}\n')

documents = []
for pdf_path in pdf_files:
    doc = process_pdf(pdf_path)
    if doc:
        documents.append(doc)

print(f'\n✅ Successfully processed {len(documents)} documents')
if documents:
    df_meta = pd.DataFrame([d['metadata'] for d in documents])
    print(df_meta[['filename','issuing_body','date','word_count']].to_string(index=False))

## 🏷️ Cell 5 — Document Type Classifier

In [ ]:
CATEGORY_KEYWORDS = {
    'Dearness Allowance / Relief' : ['dearness allowance', 'dearness relief', 'DA', 'basic pension', 'pensioners'],
    'PhD / Research Policy'       : ['ph.d', 'doctoral', 'thesis', 'research degree', 'excellence citation', 'dissertation'],
    'Internship / Training'       : ['internship', 'research internship', 'undergraduate internship', 'placement', 'apprenticeship'],
    'AI / Technology Policy'      : ['artificial intelligence', 'IndiaAI', 'AI impact', 'machine learning', 'Al mission'],
    'Student Affairs / NEP'       : ['NEP', 'national education policy', 'SAARTHI', 'student ambassador', 'curriculum'],
    'Disaster Management'         : ['disaster', 'DRR', 'disaster risk', 'NDMA', 'NIDM', 'flood', 'earthquake'],
    'Election / Democracy'        : ['election', 'IIIDEM', 'democracy', 'election management', 'voter'],
    'Accreditation / Quality'     : ['NAAC', 'accreditation', 'quality assurance', 'NIRF', 'ranking'],
    'Scholarship / Finance'       : ['scholarship', 'fellowship', 'financial assistance', 'stipend'],
    'Journalism / Media'          : ['journalism', 'syllabus', 'media', 'survey'],
    'General Circular'            : ['circular', 'notification', 'notice'],
}

def classify_document(text):
    """Classify document into a predefined category using keyword matching."""
    text_lower = text.lower()
    scores = {}
    for category, keywords in CATEGORY_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw.lower() in text_lower)
        if score > 0:
            scores[category] = score

    if not scores:
        return 'General Circular'
    return max(scores, key=scores.get)

# Classify all loaded documents
for doc in documents:
    doc['category'] = classify_document(doc['text'])
    doc['metadata']['category'] = doc['category']

print('📋 Document Classification Results:')
print('-' * 60)
for doc in documents:
    name = Path(doc['path']).name[:45]
    print(f"  {name:<48} → {doc['category']}")

## 🤖 Cell 6 — Gemini AI Analysis Pipeline

In [ ]:
def call_gemini(prompt, max_tokens=1500):
    """Call Gemini API with error handling and rate limit retry."""
    for attempt in range(3):
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens=max_tokens,
                    temperature=0.3,
                )
            )
            return response.text.strip()
        except Exception as e:
            if 'quota' in str(e).lower() or 'rate' in str(e).lower():
                print(f'  Rate limit hit, waiting 30s... (attempt {attempt+1})')
                time.sleep(30)
            else:
                print(f'  Gemini error: {e}')
                return f'Error: {e}'
    return 'Max retries reached — please check your API quota.'

def generate_plain_summary(text, category, issuing_body):
    """Generate a 10–20 line plain-language summary."""
    prompt = f"""You are an education policy expert. Analyze the following {category} document from {issuing_body}.

Write a plain-language summary (10-15 lines) that a student, faculty member, or college administrator 
can easily understand WITHOUT any policy expertise.

Your summary must answer:
1. What does this document propose or change?
2. Why was it issued — what problem does it solve?
3. Who is directly affected?
4. What action (if any) must institutions or individuals take?
5. What is the key deadline or effective date?

Write in clear, simple English. No jargon. No bullet points — use short paragraphs.

Document Text:
{text[:6000]}
"""
    return call_gemini(prompt)

def analyze_stakeholder_impact(text, category):
    """Map who benefits and who faces constraints."""
    prompt = f"""Analyze this {category} education regulation document and provide a structured stakeholder impact analysis.

For EACH of the following stakeholders, explain the impact in 2-3 sentences:
1. Students (UG / PG / PhD)
2. Faculty & Teaching Staff
3. College / University Administration
4. Academic Administrators & Registrars
5. Government / Regulatory Bodies
6. Accreditation / Compliance Teams
7. Research Scholars

Format as:
STAKEHOLDER: [name]
IMPACT TYPE: [Positive / Negative / Neutral / Mixed]
DETAILS: [2-3 sentence explanation]

Document:
{text[:5000]}
"""
    return call_gemini(prompt)

def assess_timeline_impact(text, category):
    """Assess short / medium / long-term impacts."""
    prompt = f"""Analyze this {category} education policy document and provide a structured timeline impact assessment.

Provide detailed analysis for each timeframe:

SHORT-TERM (0-12 months):
- Immediate compliance requirements
- Administrative actions needed
- Impact on current academic year

MEDIUM-TERM (1-5 years):
- Structural changes in institutions
- Curriculum or process modifications
- Workforce / faculty implications

LONG-TERM (5+ years):
- Systemic transformation expected
- Quality and outcome improvements
- Policy evolution implications

RISKS & UNINTENDED CONSEQUENCES:
- List 3-5 potential risks or challenges

Document:
{text[:5000]}
"""
    return call_gemini(prompt)

def extract_positives_negatives(text, category):
    """Extract positives and negatives in bullet format."""
    prompt = f"""For this {category} education regulation document, provide a balanced analysis.

POSITIVES (5-7 points):
List benefits, opportunities, and improvements this regulation brings.

NEGATIVES / CHALLENGES (4-6 points):
List implementation challenges, burdens, risks, or concerns.

OPPORTUNITIES (3-4 points):
What new possibilities does this open for institutions or individuals?

Use clear, simple bullet points. Each point should be 1-2 sentences.

Document:
{text[:5000]}
"""
    return call_gemini(prompt)

def build_chronology(text, category, issuing_body):
    """Identify historical context, predecessor policies, and amendments."""
    prompt = f"""Analyze this {category} document from {issuing_body} and extract policy chronology information.

Identify and explain:
1. PREDECESSOR POLICIES: Any earlier circulars, regulations, or policies this references or amends
2. LEGAL FRAMEWORK: Acts, rules, or national policies this is based on (e.g., NEP 2020, UGC Act)
3. AMENDMENT HISTORY: Any revision to previous rates, rules, or procedures mentioned
4. RELATED POLICIES: Other connected circulars or guidelines mentioned
5. POLICY CONTEXT: What broader national agenda or directive triggered this?

If specific details are not in the text, clearly state 'Not mentioned in document'.

Document:
{text[:5000]}
"""
    return call_gemini(prompt)

def detect_risks_sentiment(text, category):
    """Detect controversial areas and implementation risks."""
    prompt = f"""Analyze this {category} education regulation for potential risks and implementation challenges.

COMPLIANCE BURDEN LEVEL: [Low / Medium / High] — explain in 1 sentence.

CONTROVERSIAL CLAUSES:
Identify any clauses that might face resistance or controversy from institutions.

IMPLEMENTATION RISKS:
List the top 3-5 practical challenges for institutions trying to comply.

INSTITUTIONAL READINESS:
What infrastructure, resources, or capabilities do institutions need to implement this?

AMBIGUOUS AREAS:
Are there any clauses that are unclear or open to interpretation?

Document:
{text[:5000]}
"""
    return call_gemini(prompt)

print('✅ All AI analysis functions defined and ready')

## 🔍 Cell 7 — Run Full Analysis on All Documents

In [ ]:
from tqdm import tqdm

def analyze_document_full(doc):
    """Run all AI modules on a single document."""
    text     = doc['text']
    category = doc.get('category', 'General Circular')
    body     = doc['metadata'].get('issuing_body', 'UGC')
    filename = Path(doc['path']).name

    print(f'\n🔍 Analyzing: {filename}')
    print(f'   Category: {category} | Issuing body: {body}')

    analysis = {
        'metadata'   : doc['metadata'],
        'category'   : category,
    }

    print('   [1/6] Generating summary...')
    analysis['summary'] = generate_plain_summary(text, category, body)

    print('   [2/6] Mapping stakeholder impact...')
    analysis['stakeholder_impact'] = analyze_stakeholder_impact(text, category)

    print('   [3/6] Assessing timeline impact...')
    analysis['timeline_impact'] = assess_timeline_impact(text, category)

    print('   [4/6] Extracting positives/negatives...')
    analysis['pros_cons'] = extract_positives_negatives(text, category)

    print('   [5/6] Building policy chronology...')
    analysis['chronology'] = build_chronology(text, category, body)

    print('   [6/6] Detecting risks and sentiment...')
    analysis['risks'] = detect_risks_sentiment(text, category)

    # Save to JSON
    out_name = Path(doc['path']).stem + '_analysis.json'
    out_path = OUTPUT_DIR / out_name
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(analysis, f, indent=2, ensure_ascii=False)

    print(f'   ✅ Analysis saved → {out_path.name}')
    return analysis

# ── Run on all documents ─────────────────────────────────────────────────────
all_analyses = []
for doc in documents:
    result = analyze_document_full(doc)
    all_analyses.append(result)
    time.sleep(2)  # avoid Gemini rate limits

print(f'\n✅ Analysis complete for {len(all_analyses)} documents')
print(f'   Results saved in: {OUTPUT_DIR}')

## 📊 Cell 8 — PDF Report Generator

In [ ]:
from fpdf import FPDF
import textwrap

class ERIAReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.set_text_color(30, 80, 160)
        self.cell(0, 10, 'ERIA - Education Regulation Impact Report', align='C', new_x='LMARGIN', new_y='NEXT')
        self.set_draw_color(30, 80, 160)
        self.line(10, 20, 200, 20)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.set_text_color(120, 120, 120)
        self.cell(0, 10, f'Page {self.page_no()} | Generated by ERIA', align='C')

    def section_title(self, title):
        self.set_font('Helvetica', 'B', 11)
        self.set_fill_color(230, 240, 255)
        self.set_text_color(20, 60, 120)
        self.cell(0, 8, f'  {title}', fill=True, new_x='LMARGIN', new_y='NEXT')
        self.ln(2)

    def body_text(self, text):
        self.set_font('Helvetica', size=9)
        self.set_text_color(40, 40, 40)
        # Handle multi-line text
        for line in text.split('\n'):
            wrapped = textwrap.wrap(line, width=110)
            if not wrapped:
                self.ln(3)
            for wline in wrapped:
                self.multi_cell(0, 5, wline)
        self.ln(3)

def generate_pdf_report(analysis):
    """Generate a downloadable PDF report for one analysis."""
    pdf = ERIAReport()
    pdf.add_page()

    meta = analysis['metadata']

    # Title block
    pdf.set_font('Helvetica', 'B', 13)
    pdf.set_text_color(20, 20, 20)
    subject = meta.get('subject', meta.get('filename', 'Education Circular'))[:100]
    pdf.multi_cell(0, 7, subject)
    pdf.ln(2)

    # Metadata table
    pdf.set_font('Helvetica', size=9)
    pdf.set_text_color(80, 80, 80)
    info_items = [
        ('Issuing Body', meta.get('issuing_body', '-')),
        ('Document No.', meta.get('doc_number', '-')),
        ('Date',         meta.get('date', '-')),
        ('Category',     analysis.get('category', '-')),
        ('Word Count',   str(meta.get('word_count', '-'))),
    ]
    for label, value in info_items:
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(35, 5, f'{label}:', )
        pdf.set_font('Helvetica', size=9)
        pdf.cell(0, 5, str(value)[:100], new_x='LMARGIN', new_y='NEXT')
    pdf.ln(5)

    # Sections
    sections = [
        ('Plain Language Summary', 'summary'),
        ('Stakeholder Impact Analysis', 'stakeholder_impact'),
        ('Timeline Impact Assessment', 'timeline_impact'),
        ('Positives, Negatives & Opportunities', 'pros_cons'),
        ('Policy Chronology & Background', 'chronology'),
        ('Risk & Compliance Analysis', 'risks'),
    ]
    for title, key in sections:
        pdf.section_title(title)
        content = analysis.get(key, 'Not available')
        pdf.body_text(content[:3000])  # cap length per section

    # Save
    fname = Path(meta['filename']).stem + '_ERIA_Report.pdf'
    out_path = REPORT_DIR / fname
    pdf.output(str(out_path))
    print(f'  📄 Report saved: {out_path.name}')
    return str(out_path)

# Generate reports for all analyses
print('Generating PDF reports...')
report_paths = [generate_pdf_report(a) for a in all_analyses]
print(f'\n✅ {len(report_paths)} reports saved to {REPORT_DIR}')

## 🖥️ Cell 9 — Gradio Dashboard

In [ ]:
import gradio as gr
import tempfile

def analyze_uploaded_pdf(pdf_file):
    """Main function called by Gradio when user uploads a PDF."""
    if pdf_file is None:
        return ('Please upload a PDF file.',) * 7

    # Process the uploaded file
    doc = process_pdf(pdf_file.name)
    if not doc:
        return ('Could not extract text from this PDF. Please try another file.',) * 7

    doc['category'] = classify_document(doc['text'])
    text     = doc['text']
    category = doc['category']
    body     = doc['metadata'].get('issuing_body', 'UGC')

    # Run all AI modules
    summary   = generate_plain_summary(text, category, body)
    stake     = analyze_stakeholder_impact(text, category)
    timeline  = assess_timeline_impact(text, category)
    pros_cons = extract_positives_negatives(text, category)
    chrono    = build_chronology(text, category, body)
    risks     = detect_risks_sentiment(text, category)

    # Metadata string
    meta = doc['metadata']
    meta_str = (
        f"📌 Category     : {category}\n"
        f"🏛️  Issuing Body : {meta.get('issuing_body', '-')}\n"
        f"📋 Doc Number   : {meta.get('doc_number', '-')}\n"
        f"📅 Date         : {meta.get('date', '-')}\n"
        f"📝 Subject      : {meta.get('subject', '-')[:120]}\n"
        f"🔢 Word Count   : {meta.get('word_count', '-')}"
    )

    return meta_str, summary, stake, timeline, pros_cons, chrono, risks

def analyze_from_url(url):
    """Download PDF from a URL and analyze it."""
    if not url.strip():
        return ('Please enter a URL.',) * 7
    if not url.lower().endswith('.pdf'):
        return ('Please enter a direct PDF URL ending in .pdf',) * 7

    tmp = tempfile.NamedTemporaryFile(suffix='.pdf', delete=False)
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        r.raise_for_status()
        tmp.write(r.content)
        tmp.flush()
        return analyze_uploaded_pdf(tmp)
    except Exception as e:
        return (f'Download failed: {e}',) * 7
    finally:
        tmp.close()

# ── Build Gradio Interface ───────────────────────────────────────────────────
with gr.Blocks(
    title='ERIA — Education Regulation Impact Analyzer',
    theme=gr.themes.Soft(),
) as demo:

    gr.Markdown("""
    # 🎓 ERIA — Education Regulation Impact Analyzer
    **Simplifying UGC / AICTE / MoE policies for students, faculty, and administrators.**
    Upload any education regulation PDF and get an instant AI-powered analysis.
    """)

    with gr.Tabs():

        # ── Tab 1: Upload PDF ──────────────────────────────────────────────
        with gr.TabItem('📤 Upload PDF'):
            with gr.Row():
                pdf_input = gr.File(label='Upload PDF (UGC Circular / Regulation / Guideline)', file_types=['.pdf'])
                analyze_btn = gr.Button('🔍 Analyze Document', variant='primary')

            meta_out = gr.Textbox(label='📋 Document Metadata', lines=6, interactive=False)
            analyze_btn.click(
                fn=analyze_uploaded_pdf,
                inputs=[pdf_input],
                outputs=[meta_out,
                         gr.Textbox(label='📝 Plain Language Summary', lines=12, interactive=False),
                         gr.Textbox(label='👥 Stakeholder Impact', lines=14, interactive=False),
                         gr.Textbox(label='📅 Timeline Impact (Short/Medium/Long Term)', lines=14, interactive=False),
                         gr.Textbox(label='✅ Positives & ❌ Negatives', lines=12, interactive=False),
                         gr.Textbox(label='🕰️ Policy Chronology', lines=10, interactive=False),
                         gr.Textbox(label='⚠️ Risk & Compliance Analysis', lines=10, interactive=False)]
            )

        # ── Tab 2: URL Input ───────────────────────────────────────────────
        with gr.TabItem('🔗 Analyze from URL'):
            with gr.Row():
                url_input = gr.Textbox(
                    label='Paste direct PDF URL (must end in .pdf)',
                    placeholder='https://www.ugc.gov.in/pdfnews/example.pdf'
                )
                url_btn = gr.Button('🔍 Fetch & Analyze', variant='primary')

            url_meta = gr.Textbox(label='📋 Document Metadata', lines=6, interactive=False)
            url_btn.click(
                fn=analyze_from_url,
                inputs=[url_input],
                outputs=[url_meta,
                         gr.Textbox(label='📝 Summary', lines=12, interactive=False),
                         gr.Textbox(label='👥 Stakeholder Impact', lines=14, interactive=False),
                         gr.Textbox(label='📅 Timeline Impact', lines=14, interactive=False),
                         gr.Textbox(label='✅❌ Pros & Cons', lines=12, interactive=False),
                         gr.Textbox(label='🕰️ Chronology', lines=10, interactive=False),
                         gr.Textbox(label='⚠️ Risks', lines=10, interactive=False)]
            )

        # ── Tab 3: About ───────────────────────────────────────────────────
        with gr.TabItem('ℹ️ About'):
            gr.Markdown("""
            ## About ERIA
            **ERIA (Education Regulation Impact Analyzer)** is an AI-powered platform that converts
            complex education policy documents into simple, stakeholder-friendly summaries.

            ### What ERIA analyzes:
            - UGC Circulars, Notices, Regulations, Guidelines
            - AICTE, NAAC, NIRF notifications
            - Ministry of Education directives
            - University-level policy documents

            ### Analysis modules:
            1. **Plain Language Summary** — What does it mean for a normal person?
            2. **Stakeholder Impact** — Who is affected and how?
            3. **Timeline Assessment** — Short / Medium / Long-term impact
            4. **Pros & Cons** — Balanced perspective
            5. **Policy Chronology** — Historical context and predecessor policies
            6. **Risk Analysis** — Compliance burden and implementation challenges

            ### Tech Stack:
            Google Gemini 1.5 Flash · PyMuPDF · pdfplumber · Gradio · Python
            """)

    gr.Markdown('*Powered by Google Gemini · Built for education policy transparency*')

# ── Launch ───────────────────────────────────────────────────────────────────
demo.launch(
    share=True,          # Creates a public link — useful for demo
    debug=False,
    show_error=True,
)

## 🚀 Cell 10 — Export & Deploy to Hugging Face Spaces

In [ ]:
# ── Step 1: Generate app.py for Hugging Face Spaces ─────────────────────────
app_py = '''
import os, re, time, requests, json, textwrap, tempfile
from pathlib import Path
import fitz
import pdfplumber
import google.generativeai as genai
import gradio as gr

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-1.5-flash")

# ── (Paste all functions from Cells 4, 5, 6 above here) ──
# process_pdf, clean_text, classify_document,
# generate_plain_summary, analyze_stakeholder_impact,
# assess_timeline_impact, extract_positives_negatives,
# build_chronology, detect_risks_sentiment,
# analyze_uploaded_pdf, analyze_from_url

# ── (Paste the Gradio demo block from Cell 9 here) ──

if __name__ == "__main__":
    demo.launch()
'''

with open('/content/app.py', 'w') as f:
    f.write(app_py)

# ── Step 2: requirements.txt ─────────────────────────────────────────────────
requirements = """pymupdf
pdfplumber
google-generativeai
gradio
beautifulsoup4
requests
lxml
spacy
nltk
fpdf2
pandas
tqdm
"""
with open('/content/requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ app.py and requirements.txt generated')
print()
print('📋 Hugging Face Spaces Deployment Steps:')
print('  1. Go to https://huggingface.co/new-space')
print('  2. Choose SDK: Gradio | Space name: eria-analyzer')
print('  3. Upload app.py and requirements.txt')
print('  4. Add Secret: GEMINI_API_KEY = your_key')
print('  5. Your app goes live in ~2 minutes!')
print()
print('📦 GitHub push commands:')
print('  git init && git add . && git commit -m "ERIA v1.0"')
print('  git remote add origin https://github.com/YOUR_USERNAME/ERIA.git')
print('  git push -u origin main')